# NBA ML betting + prediction markets
Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

Odds sources: `kalshi`, `polymarket` (Polymarket US), or a sportsbook via SBR (`fanduel`, `draftkings`, ...).

## Setup

In [ ]:
%cd /content
!rm -rf /content/nba && git clone --depth 1 --branch feat/prior-season-blend https://github.com/fciaf420/NBA-Machine-Learning-Sports-Betting.git /content/nba
%cd /content/nba
!git checkout feat/prior-season-blend
!nvidia-smi -L
!pip -q install colorama sbrscrape toml xgboost==2.0.0
# Colab ships Keras 3; the saved NN models need the Keras 2 shim
!TFV=$(python -c "import tensorflow as tf; print('.'.join(tf.__version__.split('.')[:2]))") && pip -q install "tf_keras~=$TFV.0"
!python -c "import tensorflow as tf, xgboost; print('TF', tf.__version__, 'GPUs', tf.config.list_physical_devices('GPU'), 'XGB', xgboost.__version__)"

## Predict today (model vs Kalshi / Polymarket US)

In [ ]:
%cd /content/nba
!python main.py -A -odds=kalshi

## Refresh data to today (stats.nba.com + SBR odds), then rebuild the training set
Takes ~15-25 min. Then download the three `.sqlite` files to train locally if you want.

In [ ]:
%cd /content/nba/src/Process-Data
!python -m Get_Data --backfill --season 2025-26
!python -m Get_Data --backfill --season 2026-27
!python -m Get_Odds_Data --backfill --season 2025-26
!python -m Get_Odds_Data --backfill --season 2026-27
!PYTHONPATH=/content/nba python -m Create_Games

## Optional: retrain XGBoost on GPU
Models are saved to `Models/XGBoost_Models`; the newest file is used for predictions.

In [ ]:
%cd /content/nba/src/Train-Models
!python -m XGBoost_Model_ML --dataset dataset_2012-27 --trials 100 --splits 5 --calibration sigmoid
!python -m XGBoost_Model_UO --dataset dataset_2012-27 --trials 100 --splits 5 --calibration sigmoid
!python -m NN_Model_ML --dataset dataset_2012-27
!python -m NN_Model_UO --dataset dataset_2012-27

## Optional: compare prior-season blend strengths
Run after data refresh. Evaluation does not save models. Inspect Data/blend_eval.csv; market/model scores use the same market-covered games.

In [ ]:
%cd /content/nba/src/Train-Models
!python -m Evaluate_Blend

## Optional: rebuild and retrain a chosen blend
Set K and R after evaluation. This updates inference config too; do not predict until retraining completes. Same k with a different r overwrites the dataset table.

In [ ]:
K = 10
R = 0.75
import toml
p = "/content/nba/config.toml"
cfg = toml.load(p)
cfg.update(blend_k=K, blend_r=R)
with open(p, "w") as f:
    toml.dump(cfg, f)
dataset = "dataset_2012-27" + (f"_k{K:g}" if K else "")
%cd /content/nba/src/Process-Data
!PYTHONPATH=/content/nba python -m Create_Games --blend-k {K} --blend-r {R}
%cd /content/nba/src/Train-Models
!python -m XGBoost_Model_ML --dataset {dataset} --trials 100 --splits 5 --calibration sigmoid
!python -m XGBoost_Model_UO --dataset {dataset} --trials 100 --splits 5 --calibration sigmoid
!python -m NN_Model_ML --dataset {dataset}
!python -m NN_Model_UO --dataset {dataset}